In [1]:
import warnings
import requests
import os
import subprocess
import torch
import pandas as pd
from tqdm.notebook import tqdm
from faster_whisper import WhisperModel, BatchedInferencePipeline

device = ('cuda' if torch.cuda.is_available() else 'cpu')

VIDEOS_PATH = '../data/cleaned/cleaned_comments_info.csv'
FILTRATED_DIR = '../data/transcription/filtrated'
AUDIO_DIR = '../data/transcription/audios'
TRANSCRIPTION_DIR = '../data/transcription/transcriptions'

warnings.filterwarnings('ignore')
torch.cuda.empty_cache()

In [2]:
df = pd.read_csv(VIDEOS_PATH)
df['video_id'].nunique()

15808

In [3]:
for folder in [FILTRATED_DIR, AUDIO_DIR, TRANSCRIPTION_DIR]:
    os.makedirs(folder, exist_ok=True)

In [4]:
def check_video_availability(video_id):
    url = f"https://www.youtube.com/oembed?url=https://www.youtube.com/watch?v={video_id}&format=json"
    response = requests.get(url, timeout=20)
    
    return response.status_code == 200

In [5]:
def save_progress(valid_videos, invalid_videos):
    """Save valid and invalid videos to CSV files under the project log folder."""
    valid_videos.to_csv(os.path.join(FILTRATED_DIR, 'valid_videos.csv'), index=False)
    invalid_videos.to_csv(os.path.join(FILTRATED_DIR, 'invalid_videos.csv'), index=False)

In [6]:
def load_progress():
    """Load progress of valid and invalid videos (if present)."""
    try:
        valid_videos = pd.read_csv(os.path.join(FILTRATED_DIR, 'valid_videos.csv'))
    except FileNotFoundError:
        valid_videos = pd.DataFrame(columns=["video_id"])

    try:
        invalid_videos = pd.read_csv(os.path.join(FILTRATED_DIR, 'invalid_videos.csv'))
    except FileNotFoundError:
        invalid_videos = pd.DataFrame(columns=["video_id"])

    return valid_videos, invalid_videos

In [7]:
valid_videos, invalid_videos = load_progress()

In [8]:
for video_id in tqdm(df['video_id'].drop_duplicates(), desc="Checking video availability"):
    if video_id not in valid_videos["video_id"].values and video_id not in invalid_videos["video_id"].values:
        status = check_video_availability(video_id)
        
        if status is not None:
            new_row = pd.DataFrame({"video_id": [video_id]})
            if status:
                valid_videos = pd.concat([valid_videos, new_row], ignore_index=True)
            else:
                invalid_videos = pd.concat([invalid_videos, new_row], ignore_index=True)
            
            save_progress(valid_videos, invalid_videos)

Checking video availability:   0%|          | 0/15808 [00:00<?, ?it/s]

In [9]:
# Configuration
audio_folder = AUDIO_DIR
log_file = os.path.join(AUDIO_DIR, 'download_progress.txt')
csv_file = os.path.join(FILTRATED_DIR, 'valid_videos.csv')

In [10]:
def load_progress():
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            return set(f.read().splitlines())
    return set()

def save_progress(video_id):
    with open(log_file, "a") as f:
        f.write(video_id + "\n")

In [ ]:
df = pd.read_csv(csv_file)
video_ids = df["video_id"].unique().astype(str).tolist()

In [12]:
completed_videos = load_progress()

In [13]:
for video_id in tqdm(video_ids, desc="Downloading audios"):
    audio_path = os.path.join(audio_folder, f"{video_id}.mp3")

    if video_id in completed_videos or os.path.exists(audio_path):
        continue

    url = f"https://www.youtube.com/watch?v={video_id}"
    tqdm.write(f"Downloading audio from {url}...")

    try:
        command = [
            "yt-dlp",
            "--cookies-from-browser", "firefox",
            "--remote-components", "ejs:github",
            "--js-runtimes", "node",
            "--sleep-requests", "2",
            "--min-sleep-interval", "5",
            "--max-sleep-interval", "10",
            "--match-filter", "duration <= 3600",
            "-f", "bestaudio",
            "--extract-audio",
            "--audio-format", "mp3",
            "-o", f"{audio_folder}/%(id)s.%(ext)s",
            url
        ]

        result = subprocess.run(
            command,
            check=True, text=True, capture_output=True
        )

        tqdm.write(f"Download completed: {audio_path}")
        save_progress(video_id)

    except subprocess.CalledProcessError as e:
        tqdm.write(f"Error downloading {video_id}: {e.stderr}")

print("Download process finished!")

Error downloading CFPtqqULiRs: WARNING: [youtube] The provided YouTube account cookies are no longer valid. They have likely been rotated in the browser as a security measure. For tips on how to effectively export YouTube cookies, refer to  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies .
ERROR: [youtube] CFPtqqULiRs: Video unavailable

Error downloading U9b4ZlHRqwk: WARNING: [youtube] The provided YouTube account cookies are no longer valid. They have likely been rotated in the browser as a security measure. For tips on how to effectively export YouTube cookies, refer to  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies .
ERROR: [youtube] U9b4ZlHRqwk: Video unavailable

Error downloading c2qaHzEZkPo: WARNING: [youtube] The provided YouTube account cookies are no longer valid. They have likely been rotated in the browser as a security measure. For tips on how to effectively export YouTube cookies, refer to  https://github.com/yt-dlp

---
## 3. Audio Transcription

Transcribes downloaded audio using the **Whisper** model (OpenAI):
- **Model**: small (run on GPU via CUDA for better performance)
- **Language**: Portuguese
- **Output**: Text files (.txt) with full transcription
- **Optimization**: After transcription, audio is deleted
- **Checkpointing**: Avoids retranscribing already processed videos

In [14]:
df = pd.read_csv(csv_file)

audio_folder = AUDIO_DIR
transcription_folder = TRANSCRIPTION_DIR
log_file = os.path.join(TRANSCRIPTION_DIR, 'transcription_log.txt')

In [15]:
def load_progress():
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            return set(f.read().splitlines())
    return set()

def save_progress(video_id):
    with open(log_file, "a") as f:
        f.write(video_id + "\n")

In [16]:
completed_videos = set(load_progress())

In [17]:
model_size = 'small'
model = WhisperModel(
    model_size_or_path=model_size,
    device=device,
    compute_type='float16'
)
batched_model = BatchedInferencePipeline(model=model)

In [18]:
for video_id in tqdm(df['video_id'].drop_duplicates(), desc="Transcribing audios"):
    audio_path = os.path.join(audio_folder, f"{video_id}.mp3")
    output_text = os.path.join(transcription_folder, f"{video_id}.txt")

    # If already transcribed, skip
    if video_id in completed_videos or os.path.exists(output_text):
        continue

    if not os.path.exists(audio_path):
        continue

    tqdm.write(f"Transcribing {audio_path}...")

    try:
        segments, info = batched_model.transcribe(
            audio_path,
            language='pt',
            vad_filter=True,
            batch_size=16
        )
        
        # Larger videos are not feasible to process due to hardware limitations
        duration = info.duration / 60
        if duration > 60:
            tqdm.write(f'Skiping video {video_id} due to its size of {duration:.0f} minutes')
            del segments
            os.remove(audio_path)
            continue

        result = ' '.join(
            segment.text.strip()
            for segment in segments
        )
        
        # No speech detected in the video
        if not result.strip():
            tqdm.write(f'Empty transcription for {video_id}')
            os.remove(audio_path)
            continue

        with open(output_text, "w", encoding="utf-8") as f:
            f.write(result)

        save_progress(video_id)
        tqdm.write(f"Transcription saved: {output_text}")
        
        os.remove(audio_path)
        tqdm.write(f"Audio removed: {audio_path}")

    except Exception as e:
        tqdm.write(f"Error transcribing {video_id}: {str(e)}")

print("Transcription process finished!")

Transcribing audios:   0%|          | 0/32106 [00:00<?, ?it/s]

Transcription process finished!
